# Skin Cancer Detection using CNN (ISIC)
EDA → CNN → Evaluation → Save model

**Kaggle:** Add dataset `jaiahuja/skin-cancer-detection`, turn on GPU (Settings → Accelerator).

In [ ]:
import os, json, random, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(tf.__version__, tf.config.list_physical_devices("GPU"))

## 1. Locate dataset

In [ ]:
# Auto-find the Train / Test folders inside /kaggle/input
train_dir = test_dir = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if "Train" in dirs and "Test" in dirs:
        train_dir = pathlib.Path(root) / "Train"
        test_dir  = pathlib.Path(root) / "Test"
        break
print("Train:", train_dir); print("Test :", test_dir)

## 2. EDA

In [ ]:
def count_images(base):
    rows = []
    for cls in sorted(os.listdir(base)):
        n = len([f for f in os.listdir(base / cls) if f.lower().endswith((".jpg",".jpeg",".png"))])
        rows.append((cls, n))
    return pd.DataFrame(rows, columns=["class", "count"])

df_train = count_images(train_dir); df_train["split"] = "Train"
df_test  = count_images(test_dir);  df_test["split"]  = "Test"
df = pd.concat([df_train, df_test])
print(df.pivot(index="class", columns="split", values="count"))
print("Total images:", df["count"].sum())

In [ ]:
# Class distribution
plt.figure(figsize=(12,5))
sns.barplot(data=df, x="class", y="count", hue="split")
plt.xticks(rotation=45, ha="right"); plt.title("Class distribution (Train vs Test)")
plt.tight_layout(); plt.show()

plt.figure(figsize=(7,7))
plt.pie(df_train["count"], labels=df_train["class"], autopct="%1.1f%%")
plt.title("Train class share"); plt.show()

In [ ]:
# Sample images per class
classes = sorted(os.listdir(train_dir))
fig, axes = plt.subplots(3, 3, figsize=(12,12))
for ax, cls in zip(axes.ravel(), classes):
    f = random.choice(os.listdir(train_dir / cls))
    ax.imshow(Image.open(train_dir / cls / f)); ax.set_title(cls); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Image size / mode stats (sampled)
sizes, modes = [], []
for cls in classes:
    for f in os.listdir(train_dir / cls)[:60]:
        im = Image.open(train_dir / cls / f)
        sizes.append(im.size); modes.append(im.mode)
s = pd.DataFrame(sizes, columns=["width","height"])
print(s.describe()); print(pd.Series(modes).value_counts())
s.plot.scatter(x="width", y="height", title="Image dimensions (sampled)"); plt.show()

In [ ]:
# Mean RGB intensity per class (sampled)
rows = []
for cls in classes:
    for f in os.listdir(train_dir / cls)[:30]:
        a = np.asarray(Image.open(train_dir / cls / f).convert("RGB")).reshape(-1,3).mean(0)
        rows.append([cls, *a])
cdf = pd.DataFrame(rows, columns=["class","R","G","B"]).groupby("class").mean()
cdf.plot(kind="bar", figsize=(12,4), color=["r","g","b"], title="Mean channel intensity per class")
plt.show()

## 3. Data pipeline

In [ ]:
IMG_SIZE = (180, 180)
BATCH = 32

train_ds = keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode="int")
val_ds = keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode="int")
test_ds = keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH, label_mode="int", shuffle=False)

class_names = train_ds.class_names
num_classes = len(class_names)
print(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

## 4. CNN model

In [ ]:
data_aug = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
], name="augmentation")

def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    return x

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_aug(inputs)
x = layers.Rescaling(1./255)(x)
for f in (32, 64, 128, 256):
    x = conv_block(x, f)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 5. Train (records train / val / test loss every epoch)

In [ ]:
class TestLoss(keras.callbacks.Callback):
    def __init__(self, data):
        super().__init__(); self.data = data; self.test_loss = []; self.test_acc = []
    def on_epoch_end(self, epoch, logs=None):
        l, a = self.model.evaluate(self.data, verbose=0)
        self.test_loss.append(l); self.test_acc.append(a)

test_cb = TestLoss(test_ds)
callbacks = [
    test_cb,
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks)

## 6. Loss & accuracy curves

In [ ]:
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(14,5))
ax[0].plot(h["loss"], label="Train loss")
ax[0].plot(h["val_loss"], label="Validation loss")
ax[0].plot(test_cb.test_loss, label="Test loss")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(h["accuracy"], label="Train acc")
ax[1].plot(h["val_accuracy"], label="Validation acc")
ax[1].plot(test_cb.test_acc, label="Test acc")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
os.makedirs("assets", exist_ok=True)
plt.savefig("assets/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Classification report & confusion matrix

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
os.makedirs("assets", exist_ok=True)
json.dump(classification_report(y_true, y_pred, target_names=class_names, output_dict=True), open("assets/metrics.json","w"))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (Test)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.savefig("assets/confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

## 8. 10 random test images: Actual vs Predicted

In [ ]:
images = np.concatenate([x.numpy() for x, _ in test_ds]).astype("uint8")
idx = np.random.choice(len(images), 10, replace=False)

plt.figure(figsize=(16,7))
for i, j in enumerate(idx):
    plt.subplot(2, 5, i+1)
    plt.imshow(images[j]); plt.axis("off")
    ok = y_true[j] == y_pred[j]
    plt.title(f"Actual: {class_names[y_true[j]]}\nPred: {class_names[y_pred[j]]} ({y_prob[j].max():.0%})",
              color="green" if ok else "red", fontsize=9)
plt.tight_layout(); plt.show()

pd.DataFrame({"actual": [class_names[y_true[j]] for j in idx],
              "predicted": [class_names[y_pred[j]] for j in idx],
              "confidence": [round(float(y_prob[j].max()),3) for j in idx]})

## 9. Save model
Download `skin_cancer_cnn.keras` and `class_names.json` from the Output tab, then upload to your GitHub repo.

In [ ]:
model.save("skin_cancer_cnn.keras")
with open("class_names.json", "w") as f: json.dump(class_names, f)
print(os.listdir("."))
import shutil; shutil.make_archive("assets", "zip", "assets")  # download assets.zip too